# E005 — Image flagship: LBP + Logistic Regression

**Local Binary Patterns** encode local texture by comparing each pixel to its 8 neighbors.
Unlike PCA (E004) which captures global appearance, LBP is robust to:
- Monotonic lighting changes (brighter/darker room)
- Mild blur or compression artifacts
- Background variation

These are exactly the conditions Burget said eval data will have.

**Pipeline:**
1. Grayscale 80×80 image
2. Compute LBP code per pixel (P=8 neighbors, R=1)
3. Divide image into 4×4 spatial grid (16 cells of 20×20 px)
4. 256-bin histogram per cell → concatenate → 4096-dim feature vector
5. L1-normalize (histograms should sum to 1)
6. Logistic regression → score = decision_function

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. LBP implementation

For each pixel, we compare it to its 8 neighbors in clockwise order.
Each comparison gives a 1 (neighbor ≥ center) or 0. The 8 bits form a
code 0–255 — the LBP value for that pixel.

```
neighbors:  [top-left, top, top-right, right, bot-right, bot, bot-left, left]
bits:       [    b0,   b1,     b2,     b3,      b4,      b5,    b6,    b7  ]
LBP code:   b0·1 + b1·2 + b2·4 + ... + b7·128
```

We then divide the image into a grid and compute a histogram of LBP codes
per cell — capturing **where** in the face certain textures appear.

In [ ]:
def compute_lbp(gray: np.ndarray) -> np.ndarray:
    """
    Compute LBP map for a grayscale image.
    Returns array of shape (H-2, W-2) with values in [0, 255].
    """
    # 8 neighbors in clockwise order starting from top-left
    offsets = [(-1,-1), (-1, 0), (-1, 1),
               ( 0, 1),
               ( 1, 1), ( 1, 0), ( 1,-1),
               ( 0,-1)]
    H, W = gray.shape
    center = gray[1:H-1, 1:W-1].astype(np.float32)
    lbp = np.zeros_like(center, dtype=np.int32)
    for bit, (dy, dx) in enumerate(offsets):
        neighbor = gray[1+dy:H-1+dy, 1+dx:W-1+dx].astype(np.float32)
        lbp += ((neighbor >= center).astype(np.int32)) << bit
    return lbp.astype(np.uint8)


def lbp_features(png_path: Path, grid: int = 4, n_bins: int = 256) -> np.ndarray:
    """
    Load PNG → grayscale → LBP → spatial grid histograms → L1-normalized vector.
    Output shape: (grid * grid * n_bins,) = (4096,) for grid=4, bins=256.
    """
    img = np.array(Image.open(png_path).convert("RGB"), dtype=np.float32)
    gray = img.mean(axis=2)              # (80, 80)
    lbp = compute_lbp(gray)             # (78, 78) after border removal

    H, W = lbp.shape
    cell_h = H // grid
    cell_w = W // grid

    hists = []
    for i in range(grid):
        for j in range(grid):
            cell = lbp[i*cell_h:(i+1)*cell_h, j*cell_w:(j+1)*cell_w]
            hist, _ = np.histogram(cell, bins=n_bins, range=(0, n_bins))
            hist = hist.astype(np.float32)
            hist /= (hist.sum() + 1e-10)  # L1 normalize per cell
            hists.append(hist)

    return np.concatenate(hists)


def find_png(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def load_features(df: pd.DataFrame, data_dir: Path) -> np.ndarray:
    return np.stack([lbp_features(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])


# Sanity check
sample = manifest.iloc[0]
feat = lbp_features(find_png(sample["stem"], DATA))
print(f"LBP feature vector shape: {feat.shape}  (4×4 grid × 256 bins = {4*4*256})")
print(f"Feature range: [{feat.min():.4f}, {feat.max():.4f}]  (L1-normalized histograms)")

## 2. LBP visualization

Let's see what the LBP map looks like — and compare target vs non-target.
Areas with high LBP code variance = rich texture (edges, hair, wrinkles).
Flat areas (smooth skin, wall background) get near-zero variance.

In [ ]:
target_sample    = manifest[manifest.label == 1].iloc[0]
nontarget_sample = manifest[manifest.label == 0].iloc[5]

def load_gray(stem, data_dir):
    img = np.array(Image.open(find_png(stem, data_dir)).convert("RGB"), dtype=np.float32)
    return img.mean(axis=2)

gray_t  = load_gray(target_sample["stem"],    DATA)
gray_nt = load_gray(nontarget_sample["stem"], DATA)
lbp_t   = compute_lbp(gray_t)
lbp_nt  = compute_lbp(gray_nt)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for row_idx, (gray, lbp, label, stem) in enumerate([
    (gray_t,  lbp_t,  "target (m431)",   target_sample["stem"]),
    (gray_nt, lbp_nt, "non-target",       nontarget_sample["stem"]),
]):
    color = COLORS["target"] if row_idx == 0 else COLORS["nontarget"]

    axes[row_idx, 0].imshow(gray, cmap="gray", vmin=0, vmax=255)
    axes[row_idx, 0].set_title(f"Grayscale — {label}", color=color, fontsize=9)
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(lbp, cmap="viridis")
    axes[row_idx, 1].set_title("LBP map", fontsize=9)
    axes[row_idx, 1].axis("off")

    hist_vals, _ = np.histogram(lbp, bins=256, range=(0, 256))
    axes[row_idx, 2].bar(range(256), hist_vals, color=color, alpha=0.7, width=1.2)
    axes[row_idx, 2].set_title("LBP histogram (full image)", fontsize=9)
    axes[row_idx, 2].set_xlabel("LBP code")
    axes[row_idx, 2].set_ylabel("Count")

plt.suptitle("LBP: grayscale → code map → histogram", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the 4×4 spatial grid on a sample image
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (gray, lbp, label) in zip(axes, [
    (gray_t,  lbp_t,  "target"),
    (gray_nt, lbp_nt, "non-target"),
]):
    ax.imshow(gray, cmap="gray", vmin=0, vmax=255, extent=[0,78,78,0])
    # Draw 4×4 grid
    cell = 78 // 4
    for k in range(1, 4):
        ax.axhline(k * cell, color="yellow", lw=1.5, alpha=0.8)
        ax.axvline(k * cell, color="yellow", lw=1.5, alpha=0.8)
    ax.set_title(f"{label} — spatial grid (4×4 cells)",
                 color=COLORS["target"] if label=="target" else COLORS["nontarget"])
    ax.axis("off")

plt.suptitle("Spatial grid: each cell gets its own 256-bin LBP histogram", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Cross-validation

No scaler needed — LBP histograms are already L1-normalized per cell.
Logistic regression with C=1.0 (moderate regularization).

In [ ]:
SEED = 67
C_LOGREG = 1.0

oof_scores = np.full(len(manifest), np.nan)
fold_results = []
fold_val_data = []

y_all = manifest["label"].to_numpy()

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"Fold {fold_id}: train={len(train_df)}, val={len(val_df)} "
          f"(target val: {(val_df.label==1).sum()})")

    X_train = load_features(train_df, DATA)
    y_train = train_df["label"].to_numpy()
    X_val   = load_features(val_df, DATA)
    y_val   = val_df["label"].to_numpy()

    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    clf.fit(X_train, y_train)

    val_scores = clf.decision_function(X_val)
    oof_scores[val_idx] = val_scores

    eer, _     = compute_eer(val_scores[y_val==1], val_scores[y_val==0])
    min_dcf, _ = compute_min_dcf(val_scores[y_val==1], val_scores[y_val==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    fold_val_data.append({"scores": val_scores, "labels": y_val})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}")

print("\nDone.")

## 4. Results

In [ ]:
results_df = pd.DataFrame(fold_results)
mean_eer   = results_df["eer"].mean()
std_eer    = results_df["eer"].std()
mean_dcf   = results_df["min_dcf"].mean()
std_dcf    = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"\nOOF overall: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr:.3f}")

print(f"\nE004 (PCA+logreg): EER = 4.49 ± 4.26%,  min-DCF = 0.0565")
print(f"E005 (LBP+logreg): EER = {mean_eer*100:.2f} ± {std_eer*100:.2f}%, min-DCF = {mean_dcf:.4f}")
print(f"Change vs E004: {(mean_eer*100 - 4.49):+.2f}% EER")

## 5. Visualizations

In [ ]:
# Score distributions per fold + OOF
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
bin_edges = np.linspace(oof_scores.min(), oof_scores.max(), 35)

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_val_data)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l==1], s[l==0])
    ax.hist(s[l==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
    ax.hist(s[l==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2, label=f"thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER = {eer_f*100:.1f}%")
    ax.set_xlabel("Score (log-odds)")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_scores[y_all==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
ax.axvline(thr, color=COLORS["green"], ls="--", lw=2, label=f"thresh={thr:.2f}")
ax.set_title(f"OOF overall  —  EER = {eer_oof*100:.1f}%")
ax.set_xlabel("Score (log-odds)")
ax.legend(fontsize=8)

plt.suptitle("E005 LBP — Score distributions per fold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ROC + DET
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)
frr_roc = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - frr_roc))

ax = axes[0]
ax.plot(fpr, tpr, color=COLORS["target"], lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1,label="chance")
ax.scatter(fpr[eer_idx], tpr[eer_idx], color=COLORS["purple"], s=100, zorder=5,
           label=f"EER = {eer_oof*100:.1f}%")
ax.annotate(f"EER\n{eer_oof*100:.1f}%",
            xy=(fpr[eer_idx], tpr[eer_idx]),
            xytext=(fpr[eer_idx]+0.1, tpr[eer_idx]-0.12),
            arrowprops=dict(arrowstyle="->", color=COLORS["purple"]),
            color=COLORS["purple"], fontsize=9)
ax.set_xlabel("FAR")
ax.set_ylabel("1 − FRR")
ax.set_title("ROC curve (OOF)")
ax.legend()

ax = axes[1]
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]
far_c = np.clip(fpr, 1e-4, 1-1e-4)
frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=COLORS["target"], lw=2, label=f"E005 LBP  EER={eer_oof*100:.1f}%")
ax.plot(tick_pos, tick_pos, "k--", lw=1, label="EER line")
ax.scatter(scipy_norm.ppf(np.clip(fpr[eer_idx],1e-4,1-1e-4)),
           scipy_norm.ppf(np.clip(frr_roc[eer_idx],1e-4,1-1e-4)),
           color=COLORS["purple"], s=100, zorder=5)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curve (probit scale)")
ax.legend()

plt.suptitle("E005 LBP — ROC and DET curves", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# E004 vs E005 comparison
e004_fold_eers = [3.47, 0.83, 9.17]
e005_fold_eers = [r["eer"] * 100 for r in fold_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bar chart
ax = axes[0]
x = np.arange(3)
w = 0.35
b1 = ax.bar(x - w/2, e004_fold_eers, w, label="E004 PCA+logreg", color=COLORS["nontarget"], alpha=0.85)
b2 = ax.bar(x + w/2, e005_fold_eers, w, label="E005 LBP+logreg",  color=COLORS["target"],    alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER: PCA vs LBP")
ax.axhline(4.49, color=COLORS["nontarget"], ls="--", lw=1.5, alpha=0.6, label="E004 mean")
ax.axhline(mean_eer*100, color=COLORS["target"],    ls="--", lw=1.5, alpha=0.6, label="E005 mean")
ax.legend(fontsize=8)

# DET overlay
ax = axes[1]
# E004 DET — approximate from known EER
# We don't have E004 scores here, so just annotate
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=COLORS["target"], lw=2, label=f"E005 LBP  EER={eer_oof*100:.1f}%")
ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET — E005")
ax.legend()

plt.suptitle("E004 PCA vs E005 LBP — image system comparison", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# LBP cell histogram heatmap — which cells are most discriminative?
# Compute mean LBP histogram per cell for target vs non-target
target_df = manifest[manifest.label == 1]
nontarget_df = manifest[manifest.label == 0].head(30)  # match size

def cell_mean_histograms(df, data_dir, grid=4, n_bins=256):
    """Return mean histogram per cell: shape (grid*grid, n_bins)."""
    all_feats = np.stack([
        lbp_features(find_png(row["stem"], data_dir), grid=grid, n_bins=n_bins)
        for _, row in df.iterrows()
    ])
    return all_feats.reshape(len(df), grid*grid, n_bins).mean(axis=0)

mean_t  = cell_mean_histograms(target_df,   DATA)
mean_nt = cell_mean_histograms(nontarget_df, DATA)
diff = np.abs(mean_t - mean_nt).sum(axis=1).reshape(4, 4)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(diff, cmap="hot", interpolation="nearest")
plt.colorbar(im, ax=ax, label="|target − non-target| histogram distance")
ax.set_xticks(range(4)); ax.set_xticklabels(["L","CL","CR","R"])
ax.set_yticks(range(4)); ax.set_yticklabels(["Top","U","L","Bot"])
ax.set_title("LBP discriminability per spatial cell\n(brighter = more target/non-target difference)")
plt.tight_layout()
plt.show()